# Variantes de denoising con cinco semillas, NSL-KDD

Compara con el control las configuraciones que lo superan con la semilla 42, además del masking mediante Dropout al 10 %, que es la variante aplicada al tráfico real. Se repite con las semillas 42, 1, 2, 3 y 4. La partición de los datos es fija, de modo que solo varía el azar del entrenamiento. Se entrena con KDDTrain+_20Percent y se evalúa con KDDTest+, que incluye tipos de ataque no vistos en entrenamiento.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
BASE = Path.home() / ".cache/kagglehub/datasets/hassan06/nslkdd/versions/1"
df_train = pd.read_csv(BASE / "KDDTrain+_20Percent.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])
df_test = pd.read_csv(BASE / "KDDTest+.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])

df_train['is_attack'] = (df_train['label'] != 'normal').astype(int)
df_test['is_attack'] = (df_test['label'] != 'normal').astype(int)
df_train['_split'] = 'train'
df_test['_split'] = 'test'
df_all = pd.concat([df_train, df_test], ignore_index=True)

X_all = pd.get_dummies(df_all.drop(columns=['label', 'is_attack', '_split']), columns=['protocol_type', 'service', 'flag'])
is_train = (df_all['_split'] == 'train').to_numpy()

X_train_df = X_all[is_train].reset_index(drop=True)
X_test_df = X_all[~is_train].reset_index(drop=True)
y_train = df_train['is_attack'].to_numpy()
y_test = df_test['is_attack'].to_numpy()
print(f"X_train: {X_train_df.shape}, X_test: {X_test_df.shape}")

X_train: (25192, 119), X_test: (22544, 119)


In [3]:
scaler = MinMaxScaler().fit(X_train_df.to_numpy())
X_train = scaler.transform(X_train_df.to_numpy())
X_test = scaler.transform(X_test_df.to_numpy())
X_train_normal = X_train[y_train == 0]

print(f"X_train: {X_train.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test.shape}")

X_train: (25192, 119), X_train_normal: (13449, 119), X_test: (22544, 119)


In [4]:
class PureMasking(layers.Layer):
    """Anula una fracción mask_rate de las columnas sin reescalar el resto, a diferencia de Dropout."""
    def __init__(self, mask_rate, **kwargs):
        super().__init__(**kwargs)
        self.mask_rate = mask_rate

    def call(self, inputs, training=None):
        if training:
            keep_mask = tf.cast(tf.random.uniform(tf.shape(inputs)) >= self.mask_rate, inputs.dtype)
            return inputs * keep_mask
        return inputs


CORRUPTION = {
    'gaussiano': layers.GaussianNoise,
    'dropout': layers.Dropout,
    'puro': PureMasking,
}


def build_dae(input_dim, hidden_units, kind=None, level=0.0):
    """Autoencoder denso con una capa de corrupción opcional a la entrada (kind=None es el control)."""
    inputs = keras.Input(shape=(input_dim,))
    x = CORRUPTION[kind](level)(inputs) if kind else inputs
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)

In [5]:
HIDDEN_UNITS = [64, 32, 16]
CONFIGS = [('control', 0.0)] + [('dropout', 0.1), ('puro', 0.1), ('puro', 0.3)]
SEEDS = [42, 1, 2, 3, 4]

results = {}
for seed in SEEDS:
    for kind, level in CONFIGS:
        tf.keras.utils.set_random_seed(seed)
        model = build_dae(X_train.shape[1], HIDDEN_UNITS, None if kind == 'control' else kind, level)
        history = model.fit(
            X_train_normal, X_train_normal,
            epochs=100,
            batch_size=64,
            validation_split=0.1,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
            verbose=0,
        )
        auc = roc_auc_score(y_test, reconstruction_error(model, X_test))
        results[(f"{kind} {level}", seed)] = auc
        print(f"seed={seed} {kind}={level}: epocas={len(history.history['loss'])}, ROC-AUC={auc:.4f}", flush=True)

seed=42 control=0.0: epocas=100, ROC-AUC=0.9406


seed=42 dropout=0.1: epocas=100, ROC-AUC=0.9359


seed=42 puro=0.1: epocas=100, ROC-AUC=0.9480


seed=42 puro=0.3: epocas=100, ROC-AUC=0.9443


seed=1 control=0.0: epocas=100, ROC-AUC=0.9256


...

seed=3 puro=0.3: epocas=100, ROC-AUC=0.9464


seed=4 control=0.0: epocas=100, ROC-AUC=0.9329


seed=4 dropout=0.1: epocas=100, ROC-AUC=0.9431


seed=4 puro=0.1: epocas=100, ROC-AUC=0.9490


seed=4 puro=0.3: epocas=100, ROC-AUC=0.9171


In [6]:
from scipy.stats import ttest_rel

auc = pd.Series(results).unstack()  # filas: configuración, columnas: semilla
print("AUC por configuracion y semilla:")
print(auc.round(4).to_string())

control = auc.loc['control 0.0']
new = [s for s in SEEDS if s != 42]
print("\nResumen frente al control de la misma semilla:")
for name, row in auc.iterrows():
    line = f"  {name:14s} media={row.mean():.4f} std={row.std(ddof=1):.4f} media 4 nuevas={row[new].mean():.4f}"
    if name != 'control 0.0':
        d = row - control
        line += (f" | dif={d.mean():+.4f} mejora en {int((d > 0).sum())}/5 ({int((d[new] > 0).sum())}/4 nuevas)"
                 f" p={ttest_rel(row, control).pvalue:.3f}")
    print(line)

AUC por configuracion y semilla:
                 1       2       3       4       42
control 0.0  0.9256  0.9040  0.9499  0.9329  0.9406
dropout 0.1  0.9373  0.9309  0.9296  0.9431  0.9359
puro 0.1     0.9398  0.9393  0.9388  0.9490  0.9480
puro 0.3     0.9441  0.9490  0.9464  0.9171  0.9443

Resumen frente al control de la misma semilla:
  control 0.0    media=0.9306 std=0.0174 media 4 nuevas=0.9281
  dropout 0.1    media=0.9354 std=0.0054 media 4 nuevas=0.9352 | dif=+0.0048 mejora en 3/5 (3/4 nuevas) p=0.584
  puro 0.1       media=0.9430 std=0.0051 media 4 nuevas=0.9417 | dif=+0.0124 mejora en 4/5 (3/4 nuevas) p=0.172
  puro 0.3       media=0.9402 std=0.0130 media 4 nuevas=0.9392 | dif=+0.0096 mejora en 3/5 (2/4 nuevas) p=0.410
